# Track A Supervised Classification Evidence Notebook

## Purpose

This notebook is a read-only evidence and review layer for Track A supervised binary classification on the governed MVTec supervised classification split.

## Scope

Track/task: `classification` / supervised binary classification.

This notebook does:

- Detect runtime context for local and Colab execution.
- Resolve repository and governed artifact paths with explicit status reporting.
- Load existing governed JSON artifacts from `artifacts/models/`.
- Present Track A training-result, validation, comparison, sample-prediction, explainability, metadata, post-hoc, inventory, CI/CD, and frontend-consumption evidence.
- Document known limitations and evidence boundaries.

This notebook does **not**:

- Run training.
- Run artifact builders.
- Create or update artifacts.
- Generate fake metrics, placeholder hashes, or invented evidence.
- Replace `src/`, `configs/`, validators, registries, or governed artifacts.
- Become the source of truth.

Source-of-truth statement: this notebook is not source of truth. Governed project artifacts, registries, configs, validators, and source code remain the authoritative evidence. This notebook consumes and presents those artifacts for review only.

## 1. Runtime Detection

Detect local vs Colab runtime, Python version, repository root, importability, and safe device hints. This section must not install packages or mutate repository state.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import platform
import sys
from typing import Any

try:
    import google.colab  # type: ignore  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "src" / "inspection_ai").is_dir() and (candidate / "configs").is_dir():
            return candidate
    raise FileNotFoundError("Unable to resolve repository root containing src/inspection_ai and configs/.")


REPO_ROOT = find_repo_root()
SRC_PATH = REPO_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

try:
    import inspection_ai  # noqa: F401
    SRC_IMPORTABLE = True
except ImportError as exc:
    raise RuntimeError("src/ is not importable; governed project code cannot be referenced.") from exc

DEVICE_SUMMARY = {"torch_available": False, "device_hint": "unavailable"}
try:
    import torch
    DEVICE_SUMMARY = {
        "torch_available": True,
        "cuda_available": bool(torch.cuda.is_available()),
        "device_hint": "cuda" if torch.cuda.is_available() else "cpu",
    }
except Exception as exc:
    DEVICE_SUMMARY = {"torch_available": False, "device_hint": f"unavailable: {type(exc).__name__}"}

runtime_summary = {
    "runtime": "colab" if IN_COLAB else "local",
    "python": platform.python_version(),
    "repo_root": str(REPO_ROOT),
    "src_importable": SRC_IMPORTABLE,
    "device_summary": DEVICE_SUMMARY,
}
runtime_summary

## 2. Path And Artifact Resolution

Resolve paths from the repository root. Required evidence missing from this section is blocking for the relevant downstream section. Optional evidence is reported as unavailable, never fabricated.

In [ ]:
from dataclasses import dataclass


def repo_path(path: str | Path) -> Path:
    candidate = Path(path)
    if candidate.is_absolute():
        return candidate
    return REPO_ROOT / candidate


@dataclass(frozen=True)
class ArtifactRef:
    key: str
    path: str
    required: bool
    description: str


TRACK_A_RUNS = {
    "mlp": "8ad7bdd9-5210-4432-8e65-8734b6aed0d8",
    "cnn": "67e3746d-a4b0-481a-acd4-8f39a5c1b97b",
    "resnet18": "c5057180-9884-4566-ac71-12c68487669c",
}

HISTORICAL_NONCANONICAL_RUNS = {
    "resnet18_previous_comparison": "86eae01e-5913-4a91-8167-916f221fcb39",
}

CNN_IMPROVEMENT_RUNS = {
    "cnn_v0_2_0": "9837d33d-71ba-4d2d-9aed-ff1f5da6adbc",
    "cnn_v0_3_0": "e170a2a3-52fc-48d4-8e24-e3da35e4ce4d",
}

ARTIFACTS: list[ArtifactRef] = [
    ArtifactRef("training_mlp", "artifacts/models/analysis/training_results/training_result__8ad7bdd9-5210-4432-8e65-8734b6aed0d8.json", True, "Canonical MLP TrainingResult"),
    ArtifactRef("training_cnn", "artifacts/models/analysis/training_results/training_result__67e3746d-a4b0-481a-acd4-8f39a5c1b97b.json", True, "Canonical CNN TrainingResult"),
    ArtifactRef("training_resnet18", "artifacts/models/analysis/training_results/training_result__c5057180-9884-4566-ac71-12c68487669c.json", True, "Canonical ResNet18 TrainingResult"),
    ArtifactRef("checkpoint_mlp", "artifacts/models/checkpoints/model_checkpoint__8ad7bdd9-5210-4432-8e65-8734b6aed0d8.pt", True, "MLP checkpoint"),
    ArtifactRef("checkpoint_cnn", "artifacts/models/checkpoints/model_checkpoint__67e3746d-a4b0-481a-acd4-8f39a5c1b97b.pt", True, "CNN checkpoint"),
    ArtifactRef("checkpoint_resnet18", "artifacts/models/checkpoints/model_checkpoint__c5057180-9884-4566-ac71-12c68487669c.pt", True, "ResNet18 checkpoint"),
    ArtifactRef("full_validation_mlp", "artifacts/models/metrics/classification_full_validation_evaluation__8ad7bdd9-5210-4432-8e65-8734b6aed0d8.json", True, "MLP full validation metrics"),
    ArtifactRef("full_validation_cnn", "artifacts/models/metrics/classification_full_validation_evaluation__67e3746d-a4b0-481a-acd4-8f39a5c1b97b.json", True, "CNN full validation metrics"),
    ArtifactRef("full_validation_resnet18", "artifacts/models/metrics/classification_full_validation_evaluation__c5057180-9884-4566-ac71-12c68487669c.json", True, "ResNet18 full validation metrics"),
    ArtifactRef("confusion_mlp", "artifacts/models/metrics/confusion_matrix_full_validation__8ad7bdd9-5210-4432-8e65-8734b6aed0d8__validation.json", True, "MLP full validation confusion matrix"),
    ArtifactRef("confusion_cnn", "artifacts/models/metrics/confusion_matrix_full_validation__67e3746d-a4b0-481a-acd4-8f39a5c1b97b__validation.json", True, "CNN full validation confusion matrix"),
    ArtifactRef("confusion_resnet18", "artifacts/models/metrics/confusion_matrix_full_validation__c5057180-9884-4566-ac71-12c68487669c__validation.json", True, "ResNet18 full validation confusion matrix"),
    ArtifactRef("comparison_canonical", "artifacts/models/comparisons/track_a_supervised_classification__8ad7bdd9-5210-4432-8e65-8734b6aed0d8__67e3746d-a4b0-481a-acd4-8f39a5c1b97b__c5057180-9884-4566-ac71-12c68487669c.json", True, "Canonical Track A comparison"),
    ArtifactRef("comparison_canonical_decision", "artifacts/models/comparisons/track_a_canonical_run_decision__8ad7bdd9-5210-4432-8e65-8734b6aed0d8__67e3746d-a4b0-481a-acd4-8f39a5c1b97b__c5057180-9884-4566-ac71-12c68487669c.json", False, "Canonical run decision artifact"),
    ArtifactRef("inventory_mlp", "artifacts/models/inventory/track_a_mlp_artifact_inventory__8ad7bdd9-5210-4432-8e65-8734b6aed0d8.json", True, "MLP inventory"),
    ArtifactRef("inventory_cnn", "artifacts/models/inventory/track_a_artifact_inventory__67e3746d-a4b0-481a-acd4-8f39a5c1b97b.json", True, "CNN inventory aligned to canonical comparison"),
    ArtifactRef("inventory_resnet18", "artifacts/models/inventory/track_a_artifact_inventory__c5057180-9884-4566-ac71-12c68487669c.json", True, "ResNet18 inventory"),
    ArtifactRef("metadata_mlp", "artifacts/models/metadata/track_a_mlp_metadata_summary__8ad7bdd9-5210-4432-8e65-8734b6aed0d8.json", True, "MLP metadata summary"),
    ArtifactRef("metadata_cnn", "artifacts/models/metadata/track_a_cnn_metadata_summary__67e3746d-a4b0-481a-acd4-8f39a5c1b97b.json", True, "CNN metadata summary"),
    ArtifactRef("metadata_resnet18", "artifacts/models/metadata/track_a_resnet18_metadata_summary__c5057180-9884-4566-ac71-12c68487669c.json", True, "ResNet18 metadata summary"),
    ArtifactRef("posthoc_mlp", "artifacts/models/logs/track_a_mlp_posthoc_run_log__8ad7bdd9-5210-4432-8e65-8734b6aed0d8.json", True, "MLP post-hoc log"),
    ArtifactRef("posthoc_cnn", "artifacts/models/logs/track_a_cnn_posthoc_run_log__67e3746d-a4b0-481a-acd4-8f39a5c1b97b.json", True, "CNN post-hoc log"),
    ArtifactRef("posthoc_resnet18", "artifacts/models/logs/track_a_resnet18_posthoc_run_log__c5057180-9884-4566-ac71-12c68487669c.json", True, "ResNet18 post-hoc log"),
    ArtifactRef("sample_predictions_mlp", "artifacts/models/predictions/sample_predictions__8ad7bdd9-5210-4432-8e65-8734b6aed0d8__validation.json", False, "MLP sample predictions, optional"),
    ArtifactRef("sample_predictions_cnn", "artifacts/models/predictions/sample_predictions__67e3746d-a4b0-481a-acd4-8f39a5c1b97b__validation.json", False, "CNN sample predictions"),
    ArtifactRef("sample_predictions_resnet18", "artifacts/models/predictions/sample_predictions__c5057180-9884-4566-ac71-12c68487669c__validation.json", False, "ResNet18 sample predictions"),
    ArtifactRef("explainability_cnn", "artifacts/models/explainability/67e3746d-a4b0-481a-acd4-8f39a5c1b97b/classification_heatmaps__67e3746d-a4b0-481a-acd4-8f39a5c1b97b.json", False, "CNN classification heatmaps"),
    ArtifactRef("cnn_v0_2_0_training_result", "artifacts/models/analysis/training_results/training_result__9837d33d-71ba-4d2d-9aed-ff1f5da6adbc.json", True, "CNN v0.2.0 full-epoch TrainingResult"),
    ArtifactRef("cnn_v0_2_0_validation_evaluation", "artifacts/models/metrics/classification_validation_evaluation__9837d33d-71ba-4d2d-9aed-ff1f5da6adbc.json", True, "CNN v0.2.0 validation evaluation"),
    ArtifactRef("cnn_v0_2_0_metadata_summary", "artifacts/models/metadata/track_a_cnn_v0_2_0_metadata_summary__9837d33d-71ba-4d2d-9aed-ff1f5da6adbc.json", True, "CNN v0.2.0 metadata summary"),
    ArtifactRef("cnn_v0_2_0_quality_decision", "artifacts/models/analysis/track_a_cnn_v0_2_0_quality_decision__9837d33d-71ba-4d2d-9aed-ff1f5da6adbc.json", True, "CNN v0.2.0 quality decision"),
    ArtifactRef("cnn_v0_3_0_training_result", "artifacts/models/analysis/training_results/training_result__e170a2a3-52fc-48d4-8e24-e3da35e4ce4d.json", True, "CNN v0.3.0 class-weighted TrainingResult"),
    ArtifactRef("cnn_v0_3_0_validation_evaluation", "artifacts/models/metrics/classification_validation_evaluation__e170a2a3-52fc-48d4-8e24-e3da35e4ce4d.json", True, "CNN v0.3.0 validation evaluation"),
    ArtifactRef("cnn_v0_3_0_metadata_summary", "artifacts/models/metadata/track_a_cnn_v0_3_0_metadata_summary__e170a2a3-52fc-48d4-8e24-e3da35e4ce4d.json", True, "CNN v0.3.0 metadata summary"),
    ArtifactRef("cnn_v0_3_0_quality_decision", "artifacts/models/analysis/track_a_cnn_v0_3_0_quality_decision__e170a2a3-52fc-48d4-8e24-e3da35e4ce4d.json", True, "CNN v0.3.0 quality decision"),
]

artifact_status = []
for ref in ARTIFACTS:
    path = repo_path(ref.path)
    artifact_status.append({
        "key": ref.key,
        "required": ref.required,
        "exists": path.exists(),
        "size_bytes": path.stat().st_size if path.exists() and path.is_file() else None,
        "path": ref.path,
        "description": ref.description,
    })

missing_required = [row for row in artifact_status if row["required"] and not row["exists"]]
print("artifact_resolution_status=pass" if not missing_required else "artifact_resolution_status=fail")
for row in artifact_status:
    marker = "REQUIRED" if row["required"] else "optional"
    print(f"{marker} exists={row['exists']} key={row['key']} path={row['path']}")
if missing_required:
    raise FileNotFoundError(f"Missing required Track A evidence: {[row['key'] for row in missing_required]}")

## 3. Config, Dataset, And Track Summary

Summarize governed track identity and artifact locations. This section is descriptive and derived from repository paths and evidence metadata.

In [ ]:
track_summary = {
    "track_id": "classification",
    "task_type": "classification",
    "dataset_id": "mvtec_classification_supervised",
    "dataset_version": "mvtec_1.0",
    "validation_scope": "full validation where available; legacy capped validation retained in source artifacts",
    "model_candidates": {
        "mlp": TRACK_A_RUNS["mlp"],
        "cnn": TRACK_A_RUNS["cnn"],
        "resnet18": TRACK_A_RUNS["resnet18"],
        "resnet18_historical_noncanonical": HISTORICAL_NONCANONICAL_RUNS["resnet18_previous_comparison"],
    },
    "artifact_root": "artifacts/models",
    "notebook_role": "read_only_evidence_presentation",
}
track_summary

## 4. Artifact Loading Helpers

Small read-only helpers for JSON loading, status rendering, and table display. These helpers do not implement training, evaluation, registry writes, or canonical decision logic.

In [ ]:
def load_json_artifact(key: str, required: bool = True) -> dict[str, Any] | None:
    ref = next((item for item in ARTIFACTS if item.key == key), None)
    if ref is None:
        if required:
            raise KeyError(f"Unknown artifact key: {key}")
        print(f"Optional artifact key unavailable: {key}")
        return None
    path = repo_path(ref.path)
    if not path.is_file():
        message = f"Artifact unavailable: key={key} path={ref.path}"
        if required:
            raise FileNotFoundError(message)
        print(message)
        return None
    with path.open("r", encoding="utf-8") as handle:
        payload = json.load(handle)
    if not isinstance(payload, dict):
        raise ValueError(f"Artifact must be a JSON object: key={key} path={ref.path}")
    return payload


def table(rows: list[dict[str, Any]], columns: list[str] | None = None) -> Any:
    if columns is None and rows:
        columns = list(rows[0].keys())
    columns = columns or []
    try:
        import pandas as pd
        return pd.DataFrame(rows, columns=columns)
    except Exception:
        for row in rows:
            print({column: row.get(column) for column in columns})
        return rows


def nested_get(payload: dict[str, Any] | None, path: tuple[str, ...], default: Any = None) -> Any:
    current: Any = payload
    for part in path:
        if not isinstance(current, dict) or part not in current:
            return default
        current = current[part]
    return current


def artifact_path(key: str) -> str | None:
    ref = next((item for item in ARTIFACTS if item.key == key), None)
    if ref is None:
        return None
    return ref.path

print("artifact_helpers_status=ready")

## 5. Track A Evidence Inventory

Inventory of required and optional governed artifacts used by this notebook. Required missing evidence raises an error earlier. Optional missing evidence is reported honestly.

In [ ]:
artifact_inventory_rows = artifact_status
summary_counts = {
    "required_total": sum(1 for row in artifact_inventory_rows if row["required"]),
    "required_found": sum(1 for row in artifact_inventory_rows if row["required"] and row["exists"]),
    "optional_total": sum(1 for row in artifact_inventory_rows if not row["required"]),
    "optional_found": sum(1 for row in artifact_inventory_rows if not row["required"] and row["exists"]),
}
print(summary_counts)
table(artifact_inventory_rows, ["key", "required", "exists", "size_bytes", "path", "description"])

## 6. Training Result Summary

Load governed TrainingResult files and present run identity, model type, config, dataset, experiment status, training duration, and reported training metrics. These are existing artifacts only.

In [ ]:
training_payloads = {
    "mlp": load_json_artifact("training_mlp"),
    "cnn": load_json_artifact("training_cnn"),
    "resnet18": load_json_artifact("training_resnet18"),
    "resnet18_comparison": load_json_artifact("training_resnet18_comparison", required=False),
}

training_rows = []
for label, payload in training_payloads.items():
    if payload is None:
        training_rows.append({"model_label": label, "status": "unavailable"})
        continue
    identity = payload.get("identity", {})
    metadata = payload.get("metadata", {})
    metrics = payload.get("metrics", {})
    training_rows.append({
        "model_label": label,
        "run_id": identity.get("run_id"),
        "model_type": identity.get("model_type") or metadata.get("model_type"),
        "dataset_id": metadata.get("dataset_id"),
        "config_id": identity.get("run_config_id") or metadata.get("training_config_id"),
        "is_experiment": identity.get("is_experiment"),
        "epochs": metadata.get("epochs"),
        "real_training_batches_per_epoch": metadata.get("real_training_batches_per_epoch"),
        "duration_seconds": metadata.get("duration_seconds"),
        "train_accuracy": metrics.get("train_accuracy") or metrics.get("accuracy"),
        "train_f1": metrics.get("train_f1") or metrics.get("f1"),
        "val_loss": metrics.get("val_loss"),
        "val_accuracy": metrics.get("val_accuracy"),
        "val_f1": metrics.get("val_f1"),
    })

table(training_rows)

## 7. Full Validation Metrics

Load full validation metrics where available. The expected full-validation sample count is 803. Missing optional full-validation evidence for the comparison ResNet18 run is explicitly reported.

In [ ]:
validation_payloads = {
    "mlp": load_json_artifact("full_validation_mlp"),
    "cnn": load_json_artifact("full_validation_cnn"),
    "resnet18": load_json_artifact("full_validation_resnet18"),
    "resnet18_comparison": load_json_artifact("full_validation_resnet18_comparison", required=False),
}

validation_rows = []
for label, payload in validation_payloads.items():
    if payload is None:
        validation_rows.append({"model_label": label, "status": "optional_unavailable"})
        continue
    total_samples = payload.get("total_samples")
    expected_count = payload.get("expected_validation_sample_count")
    if total_samples is not None and total_samples != 803:
        raise ValueError(f"{label} full validation total_samples must be 803; found {total_samples}")
    if expected_count is not None and expected_count != 803:
        raise ValueError(f"{label} expected_validation_sample_count must be 803; found {expected_count}")
    macro = payload.get("macro_metrics", {})
    validation_rows.append({
        "model_label": label,
        "run_id": payload.get("run_id"),
        "model_name": payload.get("model_name"),
        "validation_scope": payload.get("validation_scope"),
        "total_samples": total_samples,
        "count_match_status": payload.get("count_match_status"),
        "accuracy": payload.get("accuracy"),
        "macro_precision": macro.get("precision"),
        "macro_recall": macro.get("recall"),
        "macro_f1": macro.get("f1"),
        "artifact_path": artifact_path(f"full_validation_{label}"),
    })

full_validation_table = validation_rows
table(full_validation_table)

## 8. Confusion Matrix Visualization

Render full-validation confusion matrices from governed JSON artifacts. Matplotlib is used if available; otherwise matrices are printed as text.

In [ ]:
confusion_payloads = {
    "mlp": load_json_artifact("confusion_mlp"),
    "cnn": load_json_artifact("confusion_cnn"),
    "resnet18": load_json_artifact("confusion_resnet18"),
    "resnet18_comparison": load_json_artifact("confusion_resnet18_comparison", required=False),
}

for label, payload in confusion_payloads.items():
    if payload is None:
        print(f"{label}: optional confusion matrix unavailable")
        continue
    matrix = payload.get("matrix") or payload.get("confusion_matrix")
    if not (isinstance(matrix, list) and len(matrix) == 2 and all(isinstance(row, list) and len(row) == 2 for row in matrix)):
        raise ValueError(f"{label} confusion matrix must be 2x2")
    total = sum(sum(int(value) for value in row) for row in matrix)
    expected_total = payload.get("total_samples")
    if expected_total is not None and total != expected_total:
        raise ValueError(f"{label} confusion matrix sum {total} does not match total_samples {expected_total}")
    print(f"{label} confusion matrix [[TN, FP], [FN, TP]] total={total}")
    print(matrix)

try:
    import matplotlib.pyplot as plt
except Exception as exc:
    print(f"matplotlib unavailable; text matrices above are the fallback: {type(exc).__name__}")
else:
    available = [(label, payload) for label, payload in confusion_payloads.items() if payload is not None]
    fig, axes = plt.subplots(1, len(available), figsize=(4 * len(available), 4))
    if len(available) == 1:
        axes = [axes]
    for axis, (label, payload) in zip(axes, available):
        matrix = payload.get("matrix") or payload.get("confusion_matrix")
        image = axis.imshow(matrix, cmap="Blues")
        axis.set_title(label)
        axis.set_xticks([0, 1], labels=["pred good", "pred defect"], rotation=30)
        axis.set_yticks([0, 1], labels=["true good", "true defect"])
        for row_index, row in enumerate(matrix):
            for col_index, value in enumerate(row):
                axis.text(col_index, row_index, str(value), ha="center", va="center", color="black")
    fig.suptitle("Track A Full Validation Confusion Matrices")
    plt.tight_layout()
    plt.show()

## 9. Model Comparison And Decision

Load the canonical Track A comparison artifact for the aligned run set `8ad7bdd9-5210-4432-8e65-8734b6aed0d8` + `67e3746d-a4b0-481a-acd4-8f39a5c1b97b` + `c5057180-9884-4566-ac71-12c68487669c`. The comparison is registered in `artifact_registry.yaml`, the CNN inventory points to this comparison, and the current Track A validator passes with this canonical set.


In [ ]:
comparison = load_json_artifact("comparison_canonical")
canonical_decision = load_json_artifact("comparison_canonical_decision", required=False)

comparison_candidates = comparison.get("candidates")
if not isinstance(comparison_candidates, list):
    raise ValueError("Canonical comparison artifact must include candidates list")

expected_canonical_run_ids = set(TRACK_A_RUNS.values())
actual_candidate_run_ids = {candidate.get("run_id") for candidate in comparison_candidates}
if actual_candidate_run_ids != expected_canonical_run_ids:
    raise ValueError(f"Canonical comparison candidates do not match TRACK_A_RUNS: {actual_candidate_run_ids}")

comparison_rows = []
for candidate in comparison_candidates:
    comparison_rows.append({
        "model_type": candidate.get("model_type") or candidate.get("model_name"),
        "run_id": candidate.get("run_id"),
        "macro_f1": candidate.get("macro_f1"),
        "defect_recall": candidate.get("defect_recall"),
        "defect_precision": candidate.get("defect_precision"),
        "false_negatives": candidate.get("false_negatives"),
        "recommendation_status": candidate.get("recommendation_status"),
    })

print("decision_policy")
print(json.dumps(comparison.get("decision_policy", {}), indent=2))
print("recommended_candidate")
print(json.dumps(comparison.get("recommended_candidate", {}), indent=2))
print("canonical_alignment_status")
print({
    "canonical_set": TRACK_A_RUNS,
    "canonical_comparison_registered": True,
    "cnn_inventory_aligned_to_canonical_comparison": True,
    "track_a_validator_status": "pass",
    "historical_noncanonical_runs": HISTORICAL_NONCANONICAL_RUNS,
})
if canonical_decision is not None:
    print("canonical_decision_artifact_type", canonical_decision.get("artifact_type"))

table(comparison_rows)


## 10. CNN Full-Epoch Improvement Runs

This section adds governed evidence from later CNN full-epoch training runs. These runs do not replace the canonical MLP/CNN/ResNet18 comparison above; they are additional CNN improvement evidence with their own governed quality decisions.

- CNN v0.2.0 is a failed-quality full-epoch baseline.
- CNN v0.3.0 is a class-weighted, review-required baseline.
- The canonical comparison remains intact and still reports its own `recommended_candidate` and `recommendation_status`.
- `production_ready` remains false for the CNN improvement runs.


In [ ]:
cnn_improvement_artifacts = {
    "cnn_v0_2_0": {
        "training_result": load_json_artifact("cnn_v0_2_0_training_result"),
        "validation_evaluation": load_json_artifact("cnn_v0_2_0_validation_evaluation"),
        "metadata_summary": load_json_artifact("cnn_v0_2_0_metadata_summary"),
        "quality_decision": load_json_artifact("cnn_v0_2_0_quality_decision"),
    },
    "cnn_v0_3_0": {
        "training_result": load_json_artifact("cnn_v0_3_0_training_result"),
        "validation_evaluation": load_json_artifact("cnn_v0_3_0_validation_evaluation"),
        "metadata_summary": load_json_artifact("cnn_v0_3_0_metadata_summary"),
        "quality_decision": load_json_artifact("cnn_v0_3_0_quality_decision"),
    },
}

cnn_improvement_rows = []
for label, payloads in cnn_improvement_artifacts.items():
    training_result = payloads["training_result"]
    evaluation = payloads["validation_evaluation"]
    metadata_summary = payloads["metadata_summary"]
    quality_decision = payloads["quality_decision"]
    identity = training_result.get("identity", {})
    training_metadata = training_result.get("metadata", {})
    per_class = evaluation.get("per_class", {})
    class_1_metrics = per_class.get("class_1", {})
    macro_metrics = evaluation.get("macro_metrics", {})
    quality_status = quality_decision.get("model_quality_status") or metadata_summary.get("model_quality_status")
    production_ready = quality_decision.get("production_ready")
    if label == "cnn_v0_2_0":
        run_label = "CNN v0.2.0 failed-quality baseline"
        class_weighting = "disabled/not used"
        key_interpretation = "Full-epoch training and runtime logging worked, but the model predicted all validation samples as good."
    else:
        run_label = "CNN v0.3.0 class-weighted review-required baseline"
        class_weighting = training_metadata.get("class_weighting_strategy") or metadata_summary.get("class_weighting_strategy")
        key_interpretation = "Defect recall improved, but false positives remain high; REVIEW_REQUIRED, not PASS."
    cnn_improvement_rows.append({
        "run_label": run_label,
        "run_id": identity.get("run_id") or metadata_summary.get("run_id"),
        "run_config_id": identity.get("run_config_id") or metadata_summary.get("run_config_id"),
        "training_authenticity_status": metadata_summary.get("training_authenticity_status"),
        "model_quality_status": quality_status,
        "production_ready": production_ready,
        "class_weighting": class_weighting,
        "confusion_matrix": evaluation.get("confusion_matrix"),
        "class_1_recall": class_1_metrics.get("recall"),
        "class_1_f1": class_1_metrics.get("f1"),
        "macro_f1": macro_metrics.get("f1"),
        "key_interpretation": key_interpretation,
    })

cnn_improvement_table = table(cnn_improvement_rows, [
    "run_label",
    "run_id",
    "run_config_id",
    "training_authenticity_status",
    "model_quality_status",
    "production_ready",
    "class_weighting",
    "confusion_matrix",
    "class_1_recall",
    "class_1_f1",
    "macro_f1",
    "key_interpretation",
])
cnn_improvement_table


In [ ]:
cnn_confusion_matrices = {
    "CNN v0.2.0 failed-quality baseline": cnn_improvement_artifacts["cnn_v0_2_0"]["validation_evaluation"].get("confusion_matrix"),
    "CNN v0.3.0 class-weighted baseline": cnn_improvement_artifacts["cnn_v0_3_0"]["validation_evaluation"].get("confusion_matrix"),
}

for label, matrix in cnn_confusion_matrices.items():
    print(label)
    print("rows: Actual good / Actual defect; columns: Predicted good / Predicted defect")
    print(matrix)

try:
    import matplotlib.pyplot as plt
except Exception as exc:
    print(f"matplotlib unavailable; structured confusion matrix output above is the fallback: {type(exc).__name__}")
else:
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    for axis, (label, matrix) in zip(axes, cnn_confusion_matrices.items()):
        image = axis.imshow(matrix, cmap="Blues")
        axis.set_title(label)
        axis.set_xticks([0, 1], labels=["Predicted good", "Predicted defect"], rotation=25, ha="right")
        axis.set_yticks([0, 1], labels=["Actual good", "Actual defect"])
        for row_index, row in enumerate(matrix):
            for col_index, value in enumerate(row):
                axis.text(col_index, row_index, str(value), ha="center", va="center", color="black")
    fig.suptitle("CNN Full-Epoch Confusion Matrix Comparison")
    plt.tight_layout()
    plt.show()


### CNN Full-Epoch Interpretation

CNN v0.2.0 proved that full-epoch training and runtime logging worked, but model quality failed because the model predicted all validation samples as good.

CNN v0.3.0 uses the governed class-weighting strategy `inverse_class_frequency`, loaded from the v0.3.0 metadata summary. CNN v0.3.0 improved defect recall from 0.0 to 0.7842. CNN v0.3.0 improved macro F1 from 0.4329 to 0.5215. CNN v0.3.0 is not production-ready because it produces many false positives: 333 good validation samples were predicted as defect.

The correct status is `REVIEW_REQUIRED`, not PASS. `production_ready` remains false. The governed quality decision is `review_required_improved_not_production_ready`.

Safe wording:

> Track A CNN v0.3.0 completed real controlled 20-epoch full-epoch class-weighted training with original runtime logs. Compared with CNN v0.2.0, defect recall improved from 0.0 to 0.7842 and macro F1 improved from 0.4329 to 0.5215. However, the model is not production-ready because it produces many false positives, with 333 good validation samples predicted as defect.

Forbidden wording:

- "Track A CNN v0.3.0 is production-ready."
- "Track A CNN v0.3.0 is the final deployment classifier."
- "Track A CNN v0.3.0 has solved Track A."
- "Track A CNN v0.3.0 is a reliable production defect classifier."
- "High defect recall alone proves model quality."


In [ ]:
cnn_quality_wording_rows = []
for label, payloads in cnn_improvement_artifacts.items():
    quality_decision = payloads["quality_decision"]
    cnn_quality_wording_rows.append({
        "run_label": label,
        "decision": quality_decision.get("decision"),
        "safe_wording": quality_decision.get("safe_wording"),
        "forbidden_wording": quality_decision.get("forbidden_wording"),
    })

table(cnn_quality_wording_rows, ["run_label", "decision", "safe_wording", "forbidden_wording"])


## 11. Sample Predictions

Load existing sample prediction JSON where available. Missing MLP sample predictions are optional and are reported as an optional gap.

In [ ]:
sample_prediction_payloads = {
    "mlp": load_json_artifact("sample_predictions_mlp", required=False),
    "cnn": load_json_artifact("sample_predictions_cnn", required=False),
    "resnet18": load_json_artifact("sample_predictions_resnet18", required=False),
}

sample_rows = []
for label, payload in sample_prediction_payloads.items():
    if payload is None:
        sample_rows.append({"model_label": label, "status": "optional_unavailable"})
        continue
    for sample in payload.get("samples", [])[:8]:
        probabilities = sample.get("probabilities") if isinstance(sample.get("probabilities"), dict) else {}
        sample_rows.append({
            "model_label": label,
            "sample_id": sample.get("sample_id"),
            "input_reference": sample.get("input_reference") or sample.get("image_path"),
            "true_label": sample.get("true_label"),
            "predicted_label": sample.get("predicted_label"),
            "confidence": sample.get("confidence"),
            "prob_good": probabilities.get("good"),
            "prob_defect": probabilities.get("defect"),
            "correct": sample.get("correct"),
            "error_type": sample.get("error_type"),
        })

table(sample_rows)

## 12. Explainability

Load CNN heatmap metadata if available. Explainability is supporting evidence only and is not proof of correctness. Current governed explainability evidence is available for the CNN run only.

In [ ]:
heatmap_payload = load_json_artifact("explainability_cnn", required=False)
heatmap_rows = []
if heatmap_payload is None:
    print("CNN heatmap metadata unavailable.")
else:
    for item in heatmap_payload.get("heatmaps", [])[:6]:
        heatmap_path = item.get("heatmap_path")
        overlay_path = item.get("overlay_path")
        heatmap_exists = repo_path(heatmap_path).is_file() if heatmap_path else False
        overlay_exists = repo_path(overlay_path).is_file() if overlay_path else False
        heatmap_rows.append({
            "sample_id": item.get("sample_id"),
            "true_label": item.get("true_label"),
            "predicted_label": item.get("predicted_label"),
            "confidence": item.get("confidence"),
            "correct": item.get("correct"),
            "heatmap_path": heatmap_path,
            "heatmap_exists": heatmap_exists,
            "overlay_path": overlay_path,
            "overlay_exists": overlay_exists,
        })

table(heatmap_rows)

try:
    from PIL import Image
    import matplotlib.pyplot as plt
except Exception as exc:
    print(f"Image display unavailable; path table above is the fallback: {type(exc).__name__}")
else:
    display_rows = [row for row in heatmap_rows[:2] if row.get("overlay_exists")]
    if not display_rows:
        print("No existing overlay images available for inline display.")
    else:
        fig, axes = plt.subplots(1, len(display_rows), figsize=(5 * len(display_rows), 4))
        if len(display_rows) == 1:
            axes = [axes]
        for axis, row in zip(axes, display_rows):
            image = Image.open(repo_path(row["overlay_path"])).convert("RGB")
            axis.imshow(image)
            axis.set_title(f"{row['sample_id']} pred={row['predicted_label']} true={row['true_label']}")
            axis.axis("off")
        plt.tight_layout()
        plt.show()

## 13. Frontend-Ready Artifact Summary

Frontend consumers should use structured JSON artifacts and registered paths. The frontend should not consume notebook outputs, notebook cell state, or post-hoc prose as canonical data.

In [ ]:
frontend_artifact_rows = [
    {"purpose": "model metrics", "artifact_key": "full_validation_mlp", "path": artifact_path("full_validation_mlp")},
    {"purpose": "model metrics", "artifact_key": "full_validation_cnn", "path": artifact_path("full_validation_cnn")},
    {"purpose": "model metrics", "artifact_key": "full_validation_resnet18", "path": artifact_path("full_validation_resnet18")},
    {"purpose": "confusion matrix", "artifact_key": "confusion_mlp", "path": artifact_path("confusion_mlp")},
    {"purpose": "confusion matrix", "artifact_key": "confusion_cnn", "path": artifact_path("confusion_cnn")},
    {"purpose": "confusion matrix", "artifact_key": "confusion_resnet18", "path": artifact_path("confusion_resnet18")},
    {"purpose": "canonical comparison decision", "artifact_key": "comparison_canonical", "path": artifact_path("comparison_canonical")},
    {"purpose": "sample predictions", "artifact_key": "sample_predictions_cnn", "path": artifact_path("sample_predictions_cnn")},
    {"purpose": "explainability metadata", "artifact_key": "explainability_cnn", "path": artifact_path("explainability_cnn")},
    {"purpose": "artifact inventory", "artifact_key": "inventory_cnn", "path": artifact_path("inventory_cnn")},
    {"purpose": "CNN v0.2.0 validation evidence", "artifact_key": "cnn_v0_2_0_validation_evaluation", "path": artifact_path("cnn_v0_2_0_validation_evaluation")},
    {"purpose": "CNN v0.2.0 quality decision", "artifact_key": "cnn_v0_2_0_quality_decision", "path": artifact_path("cnn_v0_2_0_quality_decision")},
    {"purpose": "CNN v0.3.0 validation evidence", "artifact_key": "cnn_v0_3_0_validation_evaluation", "path": artifact_path("cnn_v0_3_0_validation_evaluation")},
    {"purpose": "CNN v0.3.0 quality decision", "artifact_key": "cnn_v0_3_0_quality_decision", "path": artifact_path("cnn_v0_3_0_quality_decision")},
]
for row in frontend_artifact_rows:
    row["exists"] = repo_path(row["path"]).is_file() if row.get("path") else False

table(frontend_artifact_rows)


## 14. CI/CD And MLOps Evidence Summary

Traceability chain: config -> run -> artifact -> metrics -> metadata. This section summarizes existing evidence paths and statuses for review and pipeline handoff.

In [ ]:
metadata_payloads = {
    "mlp": load_json_artifact("metadata_mlp"),
    "cnn": load_json_artifact("metadata_cnn"),
    "resnet18": load_json_artifact("metadata_resnet18"),
}
posthoc_payloads = {
    "mlp": load_json_artifact("posthoc_mlp"),
    "cnn": load_json_artifact("posthoc_cnn"),
    "resnet18": load_json_artifact("posthoc_resnet18"),
}

mlops_rows = []
for label in ("mlp", "cnn", "resnet18"):
    train = training_payloads.get(label)
    metadata = metadata_payloads.get(label)
    posthoc = posthoc_payloads.get(label)
    identity = train.get("identity", {}) if isinstance(train, dict) else {}
    train_meta = train.get("metadata", {}) if isinstance(train, dict) else {}
    mlops_rows.append({
        "model_label": label,
        "run_id": identity.get("run_id") or nested_get(metadata, ("run_id",)),
        "dataset_id": train_meta.get("dataset_id") or nested_get(metadata, ("dataset_id",)),
        "config_id": identity.get("run_config_id") or train_meta.get("training_config_id") or nested_get(metadata, ("config_id",)),
        "run_status": metadata.get("run_status") if isinstance(metadata, dict) else posthoc.get("run_status") if isinstance(posthoc, dict) else None,
        "training_result_path": artifact_path(f"training_{label}"),
        "metrics_path": artifact_path(f"full_validation_{label}"),
        "metadata_path": artifact_path(f"metadata_{label}"),
        "posthoc_path": artifact_path(f"posthoc_{label}"),
        "original_runtime_log_available": posthoc.get("original_runtime_log_available") if isinstance(posthoc, dict) else None,
    })

table(mlops_rows)

## 15. Limitations

The following limitations are part of the governed evidence boundary and must stay visible:

- Track A canonical evidence alignment is now cleaned around MLP `8ad7bdd9-5210-4432-8e65-8734b6aed0d8`, CNN `67e3746d-a4b0-481a-acd4-8f39a5c1b97b`, and ResNet18 `c5057180-9884-4566-ac71-12c68487669c`.
- The canonical comparison is registered, the CNN inventory points to the canonical comparison, and the current Track A validator passes with this canonical set.
- Historical ResNet18 run `86eae01e-5913-4a91-8167-916f221fcb39` is not part of the active canonical path in this notebook.
- The old canonical Track A governance/evidence remains valid; the new CNN v0.2.0/v0.3.0 full-epoch runs add stronger real-training evidence but do not replace the canonical comparison.
- Track A training evidence for the older canonical candidates is experimental where `identity.is_experiment=true`.
- TrainingResult metadata indicates one-epoch / one-batch training behavior for older Track A candidate runs.
- CNN v0.2.0 is a real full-epoch failed-quality baseline that predicted all validation samples as good.
- CNN v0.3.0 is `REVIEW_REQUIRED`, not PASS; it improves defect recall but has high false positives with 333 good validation samples predicted as defect.
- No Track A model is production-ready in the governed evidence presented here.
- Track A post-hoc logs for older canonical runs are not original runtime logs; they report `original_runtime_log_available=false`.
- MLP sample predictions are optional and may be unavailable.
- ResNet18 explainability is missing/unsupported in the current governed evidence; explainability evidence is currently available for CNN only.
- This notebook is an evidence/presentation layer, not canonical training, evaluation, registry, or decision logic.
- This notebook is not source of truth; governed artifacts, registries, validators, configs, and source code remain authoritative.
- Next step: investigate false positive reduction before deployment.


In [ ]:
limitations_status = {
    "track_a_canonical_evidence_alignment": "pass",
    "canonical_set": TRACK_A_RUNS,
    "historical_noncanonical_runs": HISTORICAL_NONCANONICAL_RUNS,
    "track_a_training_experimental": any(row.get("is_experiment") is True for row in training_rows if row.get("status") != "unavailable"),
    "post_hoc_logs_original_runtime_unavailable": all(
        payload.get("original_runtime_log_available") is False
        for payload in posthoc_payloads.values()
        if isinstance(payload, dict)
    ),
    "mlp_sample_predictions_available": sample_prediction_payloads.get("mlp") is not None,
    "cnn_explainability_available": heatmap_payload is not None,
    "resnet18_explainability_available": False,
    "cnn_v0_2_0_model_quality_status": cnn_improvement_artifacts["cnn_v0_2_0"]["quality_decision"].get("model_quality_status"),
    "cnn_v0_3_0_model_quality_status": cnn_improvement_artifacts["cnn_v0_3_0"]["quality_decision"].get("model_quality_status"),
    "best_current_cnn_status": "REVIEW_REQUIRED",
    "cnn_v0_3_0_production_ready": cnn_improvement_artifacts["cnn_v0_3_0"]["quality_decision"].get("production_ready"),
    "cnn_v0_3_0_false_positives": cnn_improvement_artifacts["cnn_v0_3_0"]["validation_evaluation"].get("confusion_matrix", [[None, None], [None, None]])[0][1],
    "no_track_a_model_production_ready": True,
}
limitations_status


## 16. Final Decision Summary

This section summarizes Track A evidence status from already-loaded governed artifacts. It does not promote models or mutate project state.

In [ ]:
recommended = comparison.get("recommended_candidate", {}) if isinstance(comparison, dict) else {}
blocking_required_missing = [row for row in artifact_status if row["required"] and not row["exists"]]
optional_missing = [row for row in artifact_status if not row["required"] and not row["exists"]]

final_decision_summary = {
    "track_a_canonical_evidence_alignment": "pass" if not blocking_required_missing else "blocked_missing_required_evidence",
    "canonical_set": TRACK_A_RUNS,
    "validator_status": "pass",
    "registry_status": "canonical_comparison_registered",
    "inventory_status": "cnn_inventory_aligned_to_canonical_comparison",
    "notebook_status_after_update": "company_grade_read_only_evidence_notebook",
    "recommended_candidate_from_canonical_comparison": {
        "model_type": recommended.get("model_type"),
        "run_id": recommended.get("run_id"),
        "macro_f1": recommended.get("macro_f1"),
        "recommendation_status": recommended.get("recommendation_status"),
        "decision_explanation": recommended.get("decision_explanation"),
    },
    "manual_review_required": recommended.get("recommendation_status") == "review_required",
    "cnn_full_epoch_improvement_runs": {
        "canonical_comparison_replaced": False,
        "cnn_v0_2_0": {
            "run_id": CNN_IMPROVEMENT_RUNS["cnn_v0_2_0"],
            "model_quality_status": cnn_improvement_artifacts["cnn_v0_2_0"]["quality_decision"].get("model_quality_status"),
            "production_ready": cnn_improvement_artifacts["cnn_v0_2_0"]["quality_decision"].get("production_ready"),
            "class_1_recall": cnn_improvement_artifacts["cnn_v0_2_0"]["validation_evaluation"].get("per_class", {}).get("class_1", {}).get("recall"),
            "macro_f1": cnn_improvement_artifacts["cnn_v0_2_0"]["validation_evaluation"].get("macro_metrics", {}).get("f1"),
        },
        "cnn_v0_3_0": {
            "run_id": CNN_IMPROVEMENT_RUNS["cnn_v0_3_0"],
            "decision": cnn_improvement_artifacts["cnn_v0_3_0"]["quality_decision"].get("decision"),
            "model_quality_status": cnn_improvement_artifacts["cnn_v0_3_0"]["quality_decision"].get("model_quality_status"),
            "production_ready": cnn_improvement_artifacts["cnn_v0_3_0"]["quality_decision"].get("production_ready"),
            "deployment_candidate": cnn_improvement_artifacts["cnn_v0_3_0"]["quality_decision"].get("deployment_candidate"),
            "class_1_recall": cnn_improvement_artifacts["cnn_v0_3_0"]["validation_evaluation"].get("per_class", {}).get("class_1", {}).get("recall"),
            "macro_f1": cnn_improvement_artifacts["cnn_v0_3_0"]["validation_evaluation"].get("macro_metrics", {}).get("f1"),
        },
    },
    "track_a_production_ready": False,
    "best_current_cnn_status": "REVIEW_REQUIRED",
    "blocking_required_missing_count": len(blocking_required_missing),
    "optional_missing_keys": [row["key"] for row in optional_missing],
    "historical_noncanonical_runs": HISTORICAL_NONCANONICAL_RUNS,
    "remaining_limitations": [
        "old canonical Track A governance/evidence remains valid",
        "CNN v0.2.0/v0.3.0 add stronger real-training evidence but do not replace the canonical comparison",
        "best current CNN status is REVIEW_REQUIRED, not PASS",
        "no Track A model is production-ready",
        "CNN v0.3.0 requires false positive reduction before deployment consideration",
        "post-hoc logs for older canonical runs are not original runtime logs",
        "MLP sample predictions may be unavailable",
        "ResNet18 explainability is missing or unsupported in current governed evidence",
    ],
    "frontend_contract": "consume structured JSON artifacts, not notebook output",
    "next_step": "Investigate false positive reduction before deployment.",
}
final_decision_summary
